# Non-Equilibrium Financial Phase Transition Detection and Irreversibility-Aware Crisis Resolution Using Financial Digital Twins

**End-to-end runnable research implementation** following the uploaded workflow:
1. Point-in-time financial dataset and crisis episode registry
2. Leakage-controlled harmonization and chronological splits
3. Irreversibility-Sensitive Financial Phase Mapper (IS-FPM)
4. Sparse Intervention-Conditioned Financial Digital Twin (SIC-FDT)
5. Hysteresis-Penalized Crisis Resolution Optimizer (HPCRO)
6. Comparative evaluation, ablation, uncertainty and computational profiling

**Run environment:** Google Colab or Python 3.10+. The notebook runs end-to-end with reproducible synthetic data by default, so it does not require API keys or private datasets. Synthetic results are a software demonstration, **not empirical evidence**. A real-data ingestion hook for FRED is included; BIS/ECB/New York Fed series should be added with their source-specific identifiers and publication-date metadata before making empirical claims.

## 0. Environment and reproducibility

In [ ]:
# If running in a fresh Colab runtime, uncomment the next line if a package is missing:
# %pip install -q numpy pandas matplotlib scikit-learn scipy torch statsmodels

import os, time, json, math, warnings, random, platform
from IPython.display import display
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             balanced_accuracy_score, precision_recall_fscore_support,
                             average_precision_score, mean_absolute_error, mean_squared_error)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/content/financial_digital_twin_outputs") if Path("/content").exists() else Path("./financial_digital_twin_outputs")
OUT.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| Device:", DEVICE)
print("Outputs:", OUT.resolve())

## 1. Stage 1 — Build the multi-source point-in-time dataset and crisis registry

The default dataset is synthetic and monthly. Each feature has source, units, frequency, coverage and publication-lag metadata. The synthetic event registry is used only to test the pipeline.

In [ ]:
def make_synthetic_financial_panel(start="1990-01-01", periods=432, seed=SEED):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=periods, freq="MS")
    n = len(dates)
    # Correlated latent stress with persistent regimes and a few documented-for-demo episodes.
    stress = np.zeros(n)
    shocks = rng.normal(0, 0.22, n)
    for t in range(1, n):
        stress[t] = 0.92 * stress[t-1] + shocks[t]
    episodes = [
        (int(n*0.25), int(n*0.28), int(n*0.32), "synthetic_episode_1"),
        (int(n*0.57), int(n*0.60), int(n*0.64), "synthetic_episode_2"),
        (int(n*0.79), int(n*0.82), int(n*0.87), "synthetic_episode_3"),
    ]
    registry = []
    phase = np.zeros(n, dtype=int)  # 0 stable, 1 transition, 2 crisis, 3 recovery
    for onset, peak, end, eid in episodes:
        onset=max(2,onset); peak=max(onset+1,peak); end=max(peak+1,min(n-1,end))
        stress[onset:end+1] += np.linspace(0.8, 2.6, end-onset+1)
        phase[max(0,onset-3):onset] = 1
        phase[onset:peak+1] = 2
        phase[peak+1:end+1] = 3
        registry.append({"event_id":eid,"start_date":dates[onset],"peak_date":dates[peak],"recovery_date":dates[end],
                         "evidence_type":"synthetic_ground_truth_for_pipeline_testing"})
    stress = (stress - np.mean(stress)) / (np.std(stress) + 1e-8)
    # Observable variables grouped by domain.
    df = pd.DataFrame({
        "date": dates,
        "funding_liquidity": 0.5 - 0.72*stress + rng.normal(0,.22,n),
        "credit_stress": 0.1 + 0.82*stress + rng.normal(0,.25,n),
        "market_instability": 0.05 + 0.95*stress + rng.normal(0,.30,n),
        "leverage_pressure": 0.15 + 0.48*stress + rng.normal(0,.25,n),
        "inflation": 2.2 + 0.25*stress + rng.normal(0,.35,n),
        "unemployment": 5.5 + 0.42*stress + rng.normal(0,.25,n),
        "yield_spread": 1.4 - 0.38*stress + rng.normal(0,.25,n),
        "credit_growth": 3.0 - 0.55*stress + rng.normal(0,.4,n),
        "latent_stress_for_demo": stress,
        "phase_state": phase,
    })
    # Publication time approximates when the observation would become available.
    df["publication_date"] = df["date"] + pd.offsets.MonthBegin(1)
    # Missingness is retained explicitly; limited forward-fill only for observable data.
    for col in ["funding_liquidity","credit_stress","market_instability","leverage_pressure",
                "inflation","unemployment","yield_spread","credit_growth"]:
        mask = rng.random(n) < 0.025
        df.loc[mask, col] = np.nan
        df[col + "_missing"] = df[col].isna().astype(int)
    return df, pd.DataFrame(registry)

raw, crisis_registry = make_synthetic_financial_panel()
metadata = pd.DataFrame([
    {"series":"funding_liquidity","source":"Synthetic demo / conceptually NY Fed or market data","units":"index","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"credit_stress","source":"Synthetic demo / conceptually BIS","units":"index","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"market_instability","source":"Synthetic demo / conceptually FRED market series","units":"index","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"leverage_pressure","source":"Synthetic demo / conceptually BIS Z.1","units":"index","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"inflation","source":"Synthetic demo / conceptually FRED/ECB","units":"percent","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"unemployment","source":"Synthetic demo / conceptually FRED/ECB","units":"percent","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"yield_spread","source":"Synthetic demo / conceptually FRED","units":"percentage points","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
    {"series":"credit_growth","source":"Synthetic demo / conceptually BIS","units":"percent","frequency":"monthly","coverage":"demo economy","publication_lag_days":30},
])
raw.to_csv(OUT/"stage1_raw_panel.csv", index=False)
crisis_registry.to_csv(OUT/"crisis_episode_registry.csv", index=False)
metadata.to_csv(OUT/"series_metadata.csv", index=False)
display(raw.head(), crisis_registry, metadata)
print("Saved Stage 1 data to", OUT)

### Optional real-data hook: FRED

Set `FRED_API_KEY` in your environment and provide valid FRED series IDs. For a rigorous vintage-aware workflow, query ALFRED vintage dates and preserve `realtime_start`/publication timestamps. This optional function does not replace source-specific ingestion for BIS, ECB or New York Fed.

In [ ]:
def download_fred_series(series_ids, start_date="1990-01-01", api_key=None):
    """Download FRED observations. Requires internet and a FRED API key.
    Returns long-form data with observation date, series_id, value and source.
    For point-in-time backtests, use ALFRED vintages rather than treating revised values as real-time data.
    """
    import requests
    api_key = api_key or os.getenv("FRED_API_KEY")
    if not api_key:
        raise ValueError("Set FRED_API_KEY before using this optional function.")
    rows = []
    for sid in series_ids:
        url = "https://api.stlouisfed.org/fred/series/observations"
        params = {"series_id":sid, "api_key":api_key, "file_type":"json",
                  "observation_start":start_date}
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        for item in response.json().get("observations", []):
            value = item.get("value")
            rows.append({"date":item["date"],"series_id":sid,
                         "value":np.nan if value in (".", None) else float(value),
                         "realtime_start":item.get("realtime_start"),
                         "realtime_end":item.get("realtime_end"),"source":"FRED"})
    return pd.DataFrame(rows)

# Example (optional): fred_long = download_fred_series(["UNRATE", "CPIAUCSL", "DGS10", "DGS2"])

## 2. Stage 2 — Harmonization, rolling features, labels and chronological splits

No backward filling or future interpolation is used. Missingness indicators are retained. Scaler statistics are fitted on training rows only.

In [ ]:
BASE_FEATURES = ["funding_liquidity","credit_stress","market_instability","leverage_pressure",
                 "inflation","unemployment","yield_spread","credit_growth"]
DOMAIN_GROUPS = {
    "liquidity":["funding_liquidity"],
    "credit":["credit_stress","credit_growth"],
    "market":["market_instability","yield_spread"],
    "leverage":["leverage_pressure"],
    "macro":["inflation","unemployment"],
}
df = raw.sort_values("date").copy()
df["date"] = pd.to_datetime(df["date"])
# Enforce point-in-time availability: values are eligible only when published.
for col in BASE_FEATURES:
    df.loc[df["publication_date"] > df["date"] + pd.offsets.MonthBegin(1), col] = np.nan
    df[col] = df[col].ffill(limit=2)  # only values already published in the past
    df[col] = df[col].fillna(df[col].expanding(min_periods=1).median())
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_change1"] = df[col].diff()
    df[f"{col}_vol6"] = df[col].rolling(6, min_periods=3).std()
df["yield_curve_spread"] = df["yield_spread"]
df["credit_growth_change"] = df["credit_growth"].diff()
FEATURES = BASE_FEATURES + [f"{c}_lag1" for c in BASE_FEATURES] + [f"{c}_change1" for c in BASE_FEATURES] + [f"{c}_vol6" for c in BASE_FEATURES]
df[FEATURES] = df[FEATURES].replace([np.inf,-np.inf], np.nan)
# Initial rolling windows have no history; use expanding training-safe fallback.
df[FEATURES] = df[FEATURES].ffill().fillna(0.0)
n = len(df)
train_end, val_end = int(n*.65), int(n*.82)
train_idx = np.arange(0, train_end)
val_idx = np.arange(train_end, val_end)
test_idx = np.arange(val_end, n)
scaler = RobustScaler(quantile_range=(25,75))
X_train = scaler.fit_transform(df.loc[train_idx, FEATURES])
X_val = scaler.transform(df.loc[val_idx, FEATURES])
X_test = scaler.transform(df.loc[test_idx, FEATURES])
X_all = np.vstack([X_train, X_val, X_test]).astype(np.float32)
y_all = df["phase_state"].to_numpy(dtype=np.int64)
split_summary = pd.DataFrame({"partition":["train","validation","test"],
                              "start":[df.date.iloc[0],df.date.iloc[train_end],df.date.iloc[val_end]],
                              "end":[df.date.iloc[train_end-1],df.date.iloc[val_end-1],df.date.iloc[-1]],
                              "rows":[len(train_idx),len(val_idx),len(test_idx)]})
df.to_csv(OUT/"stage2_harmonized_features.csv", index=False)
split_summary.to_csv(OUT/"chronological_splits.csv", index=False)
display(split_summary)
print("Features:", len(FEATURES), "| X:", X_all.shape, "| class counts:", dict(zip(*np.unique(y_all, return_counts=True))))

## 3. Stage 3 — IS-FPM: group-adaptive gates, causal dilated convolutions and persistence-aware detection

In [ ]:
# Group indices map the original variables to domain gates. Engineered features inherit the domain of their base variable.
domain_names = list(DOMAIN_GROUPS.keys())
domain_feature_indices = {}
for domain, cols in DOMAIN_GROUPS.items():
    ids = []
    for j, feat in enumerate(FEATURES):
        base = next((c for c in cols if feat == c or feat.startswith(c+"_")), None)
        if base is not None: ids.append(j)
    domain_feature_indices[domain] = sorted(set(ids))
# Ensure every feature belongs to a group, with a catch-all for any future additions.
assigned = set(i for ids in domain_feature_indices.values() for i in ids)
domain_feature_indices["other"] = [i for i in range(len(FEATURES)) if i not in assigned]
domain_names = list(domain_feature_indices.keys())

class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1):
        super().__init__()
        self.pad = (kernel_size-1)*dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)
    def forward(self, x):
        return self.conv(nn.functional.pad(x, (self.pad, 0)))

class ISFPM(nn.Module):
    def __init__(self, n_features, domain_feature_indices, hidden=32, latent=8, n_classes=4):
        super().__init__()
        self.domain_feature_indices = domain_feature_indices
        self.gate_logits = nn.Parameter(torch.zeros(len(domain_feature_indices)))
        self.tcn1 = CausalConv1d(n_features, hidden, 3, dilation=1)
        self.tcn2 = CausalConv1d(hidden, hidden, 3, dilation=2)
        self.compress = nn.Conv1d(hidden, latent, 1)
        self.classifier = nn.Linear(latent, n_classes)
        self.forecast = nn.Linear(latent, n_features)
    def forward(self, x):
        # x shape: batch x time x features
        gates = torch.sigmoid(self.gate_logits)
        xg = x.clone()
        for g, (name, ids) in zip(gates, self.domain_feature_indices.items()):
            if ids: xg[:, :, ids] = xg[:, :, ids] * g
        z = xg.transpose(1,2)
        h = torch.relu(self.tcn1(z))
        h = torch.relu(self.tcn2(h))
        latent_seq = self.compress(h)
        latent = latent_seq[:, :, -1]
        return self.classifier(latent), latent, self.forecast(latent).squeeze(-1)

def make_windows(X, y, indices, window=12):
    # Window must end at an index inside the specified chronological partition.
    Xw, yw, end_indices = [], [], []
    allowed = set(indices.tolist())
    for end in indices:
        start = end-window+1
        if start < 0 or not all(k in allowed for k in range(start,end+1)): continue
        Xw.append(X[start:end+1]); yw.append(y[end]); end_indices.append(end)
    return np.asarray(Xw,dtype=np.float32), np.asarray(yw,dtype=np.int64), np.asarray(end_indices,dtype=int)

WINDOW = 12
Xtr, ytr, itrain = make_windows(X_all,y_all,train_idx,WINDOW)
Xva, yva, ival = make_windows(X_all,y_all,val_idx,WINDOW)
Xte, yte, itest = make_windows(X_all,y_all,test_idx,WINDOW)
print("Windowed shapes:", Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y): self.X=torch.tensor(X); self.y=torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i], self.y[i]

model = ISFPM(len(FEATURES), domain_feature_indices).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(WindowDataset(Xtr,ytr), batch_size=32, shuffle=True)
val_loader = DataLoader(WindowDataset(Xva,yva), batch_size=64, shuffle=False)

def evaluate_classifier(model, X, y):
    model.eval()
    with torch.no_grad():
        logits, _, _ = model(torch.tensor(X,device=DEVICE))
        pred = logits.argmax(1).cpu().numpy()
        probs = torch.softmax(logits,1).cpu().numpy()
    return pred, probs

history=[]
best_state=None; best_val=-1; patience=12; bad=0
start_train=time.time()
for epoch in range(60):
    model.train(); losses=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad()
        logits, latent, forecast = model(xb)
        # Phase classification + next-step feature reconstruction/prediction auxiliary objective.
        cls_loss=criterion(logits,yb)
        target=xb[:,-1,:]
        pred_loss=nn.functional.mse_loss(forecast,target)
        loss=cls_loss + 0.05*pred_loss
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        losses.append(loss.item())
    if len(Xva):
        pva,_=evaluate_classifier(model,Xva,yva)
        val_score=f1_score(yva,pva,average="macro",zero_division=0)
    else: val_score=0
    history.append({"epoch":epoch+1,"train_loss":float(np.mean(losses)),"val_macro_f1":val_score})
    if val_score>best_val:
        best_val=val_score; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
    else:
        bad+=1
    if bad>=patience: break
if best_state: model.load_state_dict(best_state)
training_seconds=time.time()-start_train
history_df=pd.DataFrame(history)
display(history_df.tail())
ax=history_df.plot(x="epoch",y=["train_loss","val_macro_f1"],secondary_y="val_macro_f1",figsize=(9,4),title="IS-FPM training")
ax.set_ylabel("Training loss"); plt.show()

### IS-FPM test evaluation and incremental change/persistence alerts

In [ ]:
pred_test, prob_test = evaluate_classifier(model, Xte, yte)
print("Balanced accuracy:", round(balanced_accuracy_score(yte,pred_test),4))
print("Macro-F1:", round(f1_score(yte,pred_test,average="macro",zero_division=0),4))
print(classification_report(yte,pred_test,labels=[0,1,2,3],
                            target_names=["stable","transition","crisis","recovery"],zero_division=0))
# Latent incremental change score and persistence encoding on all windows, calculated sequentially.
model.eval()
with torch.no_grad():
    _, latent_test, _ = model(torch.tensor(Xte,device=DEVICE))
latent_test=latent_test.cpu().numpy()
change_score=np.zeros(len(latent_test))
if len(latent_test)>1:
    change_score[1:]=np.linalg.norm(np.diff(latent_test,axis=0),axis=1)
# Threshold set from training latent change scores only.
with torch.no_grad():
    _, latent_train, _ = model(torch.tensor(Xtr,device=DEVICE))
lt=latent_train.cpu().numpy()
train_changes=np.linalg.norm(np.diff(lt,axis=0),axis=1) if len(lt)>1 else np.array([0.])
threshold=float(np.quantile(train_changes,.90))
elevated=change_score>threshold
persistence=np.convolve(elevated.astype(int),np.ones(3,dtype=int),mode="full")[:len(elevated)]
alerts=(persistence>=2)
phase_results=pd.DataFrame({"date":df.date.iloc[itest].to_numpy(),"true_phase":yte,
                            "predicted_phase":pred_test,"change_score":change_score,
                            "change_threshold":threshold,"persistent_alert":alerts})
phase_results.to_csv(OUT/"is_fpm_test_predictions.csv",index=False)
plt.figure(figsize=(12,3))
plt.plot(phase_results.date,phase_results.change_score,label="latent state-change score")
plt.axhline(threshold,linestyle="--",label="training-derived threshold")
plt.legend(); plt.title("IS-FPM incremental state-change score"); plt.tight_layout(); plt.show()

## 4. Stage 4 — SIC-FDT: sparse, stability-constrained dynamics and scenario rollout

The digital twin is a conditional scenario simulator. Its intervention outputs are not causal estimates unless a defensible causal identification strategy is supplied.

In [ ]:
SIM_VARS = ["funding_liquidity","credit_stress","market_instability","leverage_pressure"]
sim_df = df[SIM_VARS].copy()
# Scale using training-only median/IQR and build lagged dynamics.
sim_median = sim_df.iloc[train_idx].median()
sim_iqr = (sim_df.iloc[train_idx].quantile(.75)-sim_df.iloc[train_idx].quantile(.25)).replace(0,1)
sim_scaled = ((sim_df-sim_median)/sim_iqr).to_numpy()
# Fit sparse VAR(1): X[t+1] = intercept + X[t] @ A, train period only.
Y = sim_scaled[1:train_end]
Xlag = sim_scaled[:train_end-1]
coef = np.linalg.lstsq(np.column_stack([np.ones(len(Xlag)),Xlag]),Y,rcond=None)[0]
intercept, A = coef[0], coef[1:].T
# Sparsify weak edges and shrink spectral radius below 0.95 for bounded rollout.
A[np.abs(A)<0.08]=0
eig=np.max(np.abs(np.linalg.eigvals(A)))
if eig>0.95: A=A*(0.95/eig)
print("Sparse dynamics matrix A:\n",np.round(A,3))
print("Nonzero edges:",int(np.count_nonzero(A)),"/",A.size,"spectral radius:",round(float(np.max(np.abs(np.linalg.eigvals(A)))),3))

def rollout(initial_state, horizon=18, shock=None, intervention=None, A=A, intercept=intercept):
    """Roll forward in scaled coordinates. shock/intervention are additive vectors per step or None."""
    state=np.array(initial_state,dtype=float).copy()
    traj=[]
    for t in range(horizon):
        state=intercept + A@state
        if shock is not None and t < len(shock): state=state+np.asarray(shock[t])
        if intervention is not None and t < len(intervention): state=state+np.asarray(intervention[t])
        traj.append(state.copy())
    return np.asarray(traj)

# Adverse shock scenarios and candidate interventions, all defined in scaled feature space.
horizon=18
initial=sim_scaled[min(test_idx[0],len(sim_scaled)-1)].copy()
zero=np.zeros((horizon,len(SIM_VARS)))
shock=np.zeros_like(zero); shock[:3,0]=-0.8; shock[:5,1]=0.45; shock[:2,2]=0.5
baseline=rollout(initial,horizon)
shocked=rollout(initial,horizon,shock=shock)
intervention_profiles={
    "liquidity_support": np.tile(np.array([0.22,-0.03,-0.02,0.0]),(horizon,1)),
    "credit_backstop": np.tile(np.array([0.03,-0.20,-0.04,-0.05]),(horizon,1)),
    "market_stabilization": np.tile(np.array([0.02,-0.03,-0.22,-0.02]),(horizon,1)),
    "combined_bounded": np.tile(np.array([0.12,-0.12,-0.12,-0.04]),(horizon,1)),
}
scenario_trajectories={"baseline":baseline,"adverse_shock":shocked}
for name, profile in intervention_profiles.items():
    scenario_trajectories[name]=rollout(initial,horizon,shock=shock,intervention=profile)
plt.figure(figsize=(10,4))
for name,traj in scenario_trajectories.items():
    stress_index=traj[:,1:].mean(axis=1)
    plt.plot(np.arange(1,horizon+1),stress_index,label=name)
plt.xlabel("Months ahead"); plt.ylabel("Mean scaled stress indicators")
plt.title("SIC-FDT reduced-order scenario rollouts"); plt.legend(ncol=2); plt.tight_layout(); plt.show()

## 5. Stage 5 — HPCRO: hysteresis-penalized intervention ranking

Candidate scores combine cumulative stress, recovery duration, persistent deviation, intervention cost and a renewed-stress penalty. We use a bounded, predefined candidate set.

In [ ]:
# Reference resilient range: training-period central region in scaled coordinates.
train_sim = sim_scaled[:train_end]
reference = np.median(train_sim,axis=0)
reference_band = np.quantile(np.abs(train_sim-reference),.60,axis=0) + 1e-6
costs={"liquidity_support":0.25,"credit_backstop":0.30,"market_stabilization":0.22,"combined_bounded":0.55}

def score_trajectory(traj, cost=0.0, band=reference_band):
    # Stress proxy: credit + market + leverage; lower is better in this constructed scenario.
    stress=np.maximum(0,traj[:,1:].mean(axis=1))
    cumulative_stress=float(np.sum(stress))
    deviation=np.maximum(0,np.abs(traj-reference)-band)
    persistent_deviation=float(np.sum(np.mean(deviation,axis=1)))
    # Recovery = first of 3 consecutive points inside the resilient band.
    inside=np.all(np.abs(traj-reference)<=band,axis=1)
    recovery=None
    for i in range(max(0,len(inside)-2)):
        if inside[i:i+3].all(): recovery=i+1; break
    recovery_duration=float(recovery if recovery is not None else len(traj)+6)
    # Renewed crisis proxy: stress rises again after an initial decline.
    renewed=float(np.sum(np.diff(stress)>0.12))
    total=1.0*cumulative_stress + 1.2*persistent_deviation + 0.35*recovery_duration + 0.8*renewed + 1.5*cost
    return {"cumulative_stress":cumulative_stress,"persistent_deviation":persistent_deviation,
            "recovery_duration_months":recovery_duration,"incomplete_recovery":recovery is None,
            "renewed_stress_penalty":renewed,"intervention_cost":cost,"hpcro_score":total}

ranking=[]
for name,traj in scenario_trajectories.items():
    row={"candidate":name,**score_trajectory(traj,cost=costs.get(name,0.0))}
    ranking.append(row)
ranking_df=pd.DataFrame(ranking).sort_values("hpcro_score").reset_index(drop=True)
display(ranking_df)
ranking_df.to_csv(OUT/"hpcro_intervention_ranking.csv",index=False)
print("Preferred candidate under this synthetic model:",ranking_df.iloc[0]["candidate"])

### Uncertainty-aware candidate ranking

A lightweight sensitivity analysis perturbs the fitted transition matrix and shock magnitude. These are model-sensitivity intervals, not statistical confidence intervals from observed crisis episodes.

In [ ]:
rng=np.random.default_rng(SEED)
uncertainty_rows=[]
for name, profile in intervention_profiles.items():
    scores=[]
    for b in range(100):
        A_b=A + rng.normal(0,0.015,A.shape)
        eig=np.max(np.abs(np.linalg.eigvals(A_b)))
        if eig>0.95: A_b=A_b*(0.95/eig)
        shock_b=shock*rng.uniform(.8,1.2)
        tr=rollout(initial,horizon,shock=shock_b,intervention=profile,A=A_b)
        scores.append(score_trajectory(tr,cost=costs[name])["hpcro_score"])
    uncertainty_rows.append({"candidate":name,"score_mean":np.mean(scores),
                             "score_p05":np.quantile(scores,.05),"score_p95":np.quantile(scores,.95),
                             "score_sd":np.std(scores)})
uncertainty_df=pd.DataFrame(uncertainty_rows).sort_values("score_mean")
display(uncertainty_df)
uncertainty_df.to_csv(OUT/"hpcro_sensitivity.csv",index=False)

## 6. Stage 6 — comparative evaluation, ablations, robustness and profiling

In [ ]:
# Baseline classifiers trained on flattened historical windows; same chronological train/test partitions.
def flatten_windows(X): return X.reshape(len(X),-1)
baseline_results=[]
if len(Xtr)>0 and len(Xte)>0:
    for name, clf in [
        ("LogisticRegression",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED)),
        ("RandomForest",RandomForestClassifier(n_estimators=250,max_depth=8,class_weight="balanced",random_state=SEED,n_jobs=-1))
    ]:
        clf.fit(flatten_windows(Xtr),ytr)
        p=clf.predict(flatten_windows(Xte))
        baseline_results.append({"model":name,"balanced_accuracy":balanced_accuracy_score(yte,p),
                                 "macro_f1":f1_score(yte,p,average="macro",zero_division=0)})
    baseline_results.append({"model":"IS-FPM","balanced_accuracy":balanced_accuracy_score(yte,pred_test),
                             "macro_f1":f1_score(yte,pred_test,average="macro",zero_division=0)})
baseline_df=pd.DataFrame(baseline_results)
display(baseline_df)
baseline_df.to_csv(OUT/"phase_detection_comparison.csv",index=False)
if len(baseline_df):
    baseline_df.set_index("model")[["balanced_accuracy","macro_f1"]].plot(kind="bar",figsize=(8,4),ylim=(0,1),title="Phase detection comparison")
    plt.tight_layout(); plt.show()

In [ ]:
# Event-level warning summary: test-period alerts compared with documented synthetic crisis starts.
test_dates=pd.to_datetime(phase_results["date"])
test_start=test_dates.min() if len(test_dates) else df.date.iloc[-1]
test_end=test_dates.max() if len(test_dates) else df.date.iloc[-1]
events_test=crisis_registry[(crisis_registry["start_date"]>=test_start)&(crisis_registry["start_date"]<=test_end)].copy()
alert_dates=pd.to_datetime(phase_results.loc[phase_results.persistent_alert,"date"])
lead_times=[]
for _,ev in events_test.iterrows():
    prior=alert_dates[alert_dates<ev.start_date]
    lead_days=(ev.start_date-prior.max()).days if len(prior) else np.nan
    lead_times.append({"event_id":ev.event_id,"start_date":ev.start_date,
                       "warned_before_onset":bool(len(prior)),"lead_days":lead_days})
event_warning_df=pd.DataFrame(lead_times)
display(event_warning_df)
event_warning_df.to_csv(OUT/"event_level_early_warnings.csv",index=False)
if len(alert_dates):
    years=max((test_end-test_start).days/365.25,1/12)
    print("Persistent alert observations:",len(alert_dates),"| alerts/year:",round(len(alert_dates)/years,2))
print("Median warning lead days:",event_warning_df.lead_days.median() if len(event_warning_df) else "No test crisis starts in the selected test period")

In [ ]:
# Component ablations on the digital twin / optimizer objective.
# 1) no hysteresis: rank only by cumulative stress
# 2) dense dynamics: refit without thresholding weak coefficients (still stability-shrunk)
coef_dense=np.linalg.lstsq(np.column_stack([np.ones(len(Xlag)),Xlag]),Y,rcond=None)[0]
A_dense=coef_dense[1:].T
eig=np.max(np.abs(np.linalg.eigvals(A_dense)))
if eig>0.95: A_dense=A_dense*(0.95/eig)
ablation_rows=[]
for name, profile in intervention_profiles.items():
    tr_sparse=rollout(initial,horizon,shock=shock,intervention=profile,A=A)
    tr_dense=rollout(initial,horizon,shock=shock,intervention=profile,A=A_dense)
    sparse_score=score_trajectory(tr_sparse,costs[name])
    dense_score=score_trajectory(tr_dense,costs[name])
    ablation_rows.extend([
        {"candidate":name,"variant":"full_SIC-FDT_plus_HPCRO","score":sparse_score["hpcro_score"]},
        {"candidate":name,"variant":"no_hysteresis_penalty","score":sparse_score["cumulative_stress"]+1.5*costs[name]},
        {"candidate":name,"variant":"dense_dynamics","score":dense_score["hpcro_score"]},
    ])
ablation_df=pd.DataFrame(ablation_rows)
display(ablation_df)
ablation_df.to_csv(OUT/"component_ablation.csv",index=False)

In [ ]:
# Controlled missingness robustness: mask a fraction of test-window features, use zero as the
# training-scaled median, and report degradation. This is a perturbation test, not real-world missingness validation.
robust_rows=[]
model.eval()
for missing_rate in [0.0,0.05,0.10,0.20]:
    Xpert=Xte.copy()
    rng=np.random.default_rng(SEED+int(missing_rate*1000))
    mask=rng.random(Xpert.shape)<missing_rate
    Xpert[mask]=0.0
    pp,_=evaluate_classifier(model,Xpert,yte)
    robust_rows.append({"masked_fraction":missing_rate,
                        "balanced_accuracy":balanced_accuracy_score(yte,pp),
                        "macro_f1":f1_score(yte,pp,average="macro",zero_division=0)})
robust_df=pd.DataFrame(robust_rows)
display(robust_df)
robust_df.to_csv(OUT/"missingness_robustness.csv",index=False)
robust_df.plot(x="masked_fraction",y=["balanced_accuracy","macro_f1"],marker="o",ylim=(0,1),figsize=(7,4),title="Missingness perturbation robustness")
plt.tight_layout(); plt.show()

## 7. Computational profile and final artifacts

In [ ]:
n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
# Warm-up and inference latency
if len(Xte):
    xb=torch.tensor(Xte[:min(64,len(Xte))],device=DEVICE)
    with torch.no_grad():
        for _ in range(3): model(xb)
    t0=time.perf_counter()
    with torch.no_grad():
        for _ in range(20): model(xb)
    latency_ms=(time.perf_counter()-t0)*1000/(20*len(xb))
else: latency_ms=np.nan
profile = {
    "device":str(DEVICE),
    "trainable_parameters":int(n_params),
    "training_seconds":float(training_seconds),
    "inference_ms_per_window":float(latency_ms),
    "feature_count":int(len(FEATURES)),
    "window_length":int(WINDOW),
    "train_windows":int(len(Xtr)),
    "validation_windows":int(len(Xva)),
    "test_windows":int(len(Xte)),
    "random_seed":SEED,
    "data_mode":"synthetic demonstration",
    "interpretation_note":"Metrics and scenario rankings are synthetic pipeline checks, not empirical financial conclusions."
}
with open(OUT/"computational_profile.json","w") as f: json.dump(profile,f,indent=2)
display(pd.DataFrame([profile]).T.rename(columns={0:"value"}))
print("\nGenerated output files:")
for p in sorted(OUT.iterdir()):
    print("-",p.name, f"({p.stat().st_size:,} bytes)")

## 8. Research and methodological checks before empirical publication

- Replace synthetic data with versioned FRED/ALFRED, BIS, ECB and New York Fed series.
- For every input, preserve observation date, release/publication timestamp, vintage/revision identifier, units, geography and missingness metadata.
- Confirm each feature at forecast origin was genuinely available at that time. The demo uses an approximate one-month publication lag and is not a substitute for true ALFRED vintage reconstruction.
- Replace synthetic crisis dates with an independently sourced crisis registry and document inclusion/overlap rules.
- Keep the final test period and crisis episodes untouched during model and threshold selection.
- Report per-class precision/recall, event-level warning recall, lead time, false alarms/year, twin forecast errors, recovery metrics, confidence intervals and computational cost.
- Use block-bootstrap intervals over contiguous time blocks / crisis episodes for empirical uncertainty.
- Treat interventions as conditional simulations unless causal identification assumptions are defensible.
- Do not describe synthetic demonstration performance as validated real-world performance.

### Outputs
The notebook saves CSV/JSON artifacts to `financial_digital_twin_outputs/`, including the raw panel, crisis registry, metadata, harmonized features, splits, IS-FPM predictions, intervention ranking, sensitivity analysis, comparisons, ablations, robustness results and computational profile.